# 10 — Replay Warehouse + Ladder Forensics (CPU)

This replaces **E000 + E001A + E001B + E002** with one restart-safe job.

## Kaggle inputs
- Required: `kaggle/kaggriculture-episodes-index`
- Code repo Dataset: optional. If absent, the notebook clones the public GitHub repo.
- Accelerator: **None**
- Internet: **ON** because public replay acquisition uses Kaggle's official simulation CLI.

## CPU design
- schema discovery before assumptions;
- current-engine filtering when timestamps exist;
- stratified episode sampling;
- concurrent replay downloads with conservative worker count;
- process-parallel replay parsing into Parquet shards;
- one final turn warehouse, one daily warehouse;
- Bradley–Terry strength and open-loop diagnostics in the same job;
- checkpoint manifests throughout so failed downloads do not erase progress.

### Default scale
`MAX_REPLAYS = 300` is a robust first production run. Increase to 1,000–3,000 after the pipeline is green.


In [ ]:
from pathlib import Path
import os,sys,subprocess,json,shutil

INPUT=Path('/kaggle/input')
WORK=Path('/kaggle/working/kagv2')
WORK.mkdir(parents=True,exist_ok=True)

def find_repo():
    roots=[INPUT,Path('/kaggle/working'),Path.cwd()]
    hit=next((p for r in roots if r.exists() for p in r.rglob('src/kagv2/__init__.py')),None)
    return hit.parents[2] if hit else None

def expected_commit():
    hits=list(INPUT.rglob('repo_commit.txt')) if INPUT.exists() else []
    if hits:
        x=hits[0].read_text().strip()
        return x if x else None
    return None

ROOT=find_repo()
if ROOT is None:
    dst=Path('/kaggle/working/kaggriculture')
    if not dst.exists():
        r=subprocess.run(
            ['git','clone','--depth','1','https://github.com/sidhulyalkar/kaggriculture.git',str(dst)],
            capture_output=True,text=True
        )
        if r.returncode:
            raise RuntimeError(
                'Could not find an attached code repo and GitHub clone failed. '
                'Turn Internet ON or attach your kaggriculture-code-repo Dataset.\n'+r.stderr[-2000:]
            )
    ROOT=dst
    ref=expected_commit()
    if ref:
        subprocess.run(['git','-C',str(ROOT),'fetch','--depth','1','origin',ref],capture_output=True,text=True)
        c=subprocess.run(['git','-C',str(ROOT),'checkout',ref],capture_output=True,text=True)
        if c.returncode:
            print('WARN: could not checkout pinned commit',ref,c.stderr[-500:])

sys.path.insert(0,str(ROOT))
sys.path.insert(0,str(ROOT/'src'))
commit=subprocess.run(['git','-C',str(ROOT),'rev-parse','HEAD'],capture_output=True,text=True).stdout.strip()
if commit:
    (WORK/'repo_commit.txt').write_text(commit)
print('ROOT =',ROOT)
print('WORK =',WORK)
print('COMMIT =',commit or 'dataset snapshot')


In [ ]:
import pandas as pd, numpy as np, time, subprocess, os
from concurrent.futures import ThreadPoolExecutor, as_completed
from src.kagv2.schema import audit_root,choose_index_table,normalize_index

EP_ROOT=Path('/kaggle/input/kaggriculture-episodes-index')
if not EP_ROOT.exists():
    hits=[p for p in INPUT.glob('*') if 'kaggriculture' in p.name.lower() and 'episode' in p.name.lower()]
    if not hits: raise FileNotFoundError('Attach kaggle/kaggriculture-episodes-index')
    EP_ROOT=hits[0]

audit=audit_root(EP_ROOT)
audit.to_csv(WORK/'episode_schema_report.csv',index=False)
display(audit)

idx_path,raw=choose_index_table(EP_ROOT)
catalog=normalize_index(raw)
print('Selected index:',idx_path,'shape=',catalog.shape)
display(catalog.head())
if 'episode_id' not in catalog:
    raise RuntimeError('Could not infer episode_id. Inspect episode_schema_report.csv before continuing.')


In [ ]:
# -------- Episode selection --------
MAX_REPLAYS=300
RNG=np.random.default_rng(20260817)
d=catalog.copy()

if 'created_at' in d:
    d=d[d.created_at.notna()].copy()
    cur=d[d.created_at>=pd.Timestamp('2026-08-07',tz='UTC')]
    if len(cur)>=50:
        d=cur
        print('Current-engine rows:',len(d))

def clean_id(x):
    if pd.isna(x): return None
    try:
        f=float(x)
        if f.is_integer(): return str(int(f))
    except Exception: pass
    s=str(x).strip()
    return s or None

d['episode_id_clean']=d['episode_id'].map(clean_id)
d=d[d.episode_id_clean.notna()].copy()

# One row per episode. Keep max rating/reward when the index is agent-granular.
agg={}
for c in d.columns:
    if c=='episode_id_clean': continue
    if c in ('rating','reward','created_at'): agg[c]='max'
    elif c in ('submission_id','team_name','team_id'): agg[c]='first'
eps=d.groupby('episode_id_clean',as_index=False).agg(agg) if agg else d[['episode_id_clean']].drop_duplicates()
eps=eps.rename(columns={'episode_id_clean':'episode_id'})
print('Candidate unique episodes:',len(eps))

chosen=[]
target=min(MAX_REPLAYS,len(eps))
if 'rating' in eps and pd.to_numeric(eps.rating,errors='coerce').notna().sum()>=20:
    eps['_rating']=pd.to_numeric(eps.rating,errors='coerce')
    chosen += eps.sort_values('_rating',ascending=False).head(max(1,target//2)).episode_id.tolist()
if 'created_at' in eps and eps.created_at.notna().sum()>=20:
    chosen += eps.sort_values('created_at',ascending=False).head(max(1,int(target*.30))).episode_id.tolist()
chosen=list(dict.fromkeys(chosen))
remaining=[x for x in eps.episode_id.tolist() if x not in set(chosen)]
need=target-len(chosen)
if need>0:
    chosen += RNG.choice(remaining,size=min(need,len(remaining)),replace=False).tolist()
chosen=list(dict.fromkeys(chosen))[:target]

selected=eps[eps.episode_id.isin(chosen)].copy()
selected.to_csv(WORK/'selected_episodes.csv',index=False)
print('Selected:',len(chosen))
display(selected.head(20))


In [ ]:
# -------- Replay acquisition --------
REPLAY_DIR=WORK/'replays'
REPLAY_DIR.mkdir(exist_ok=True)
DOWNLOAD_WORKERS=min(4,max(1,(os.cpu_count() or 2)))
RETRIES=3

# Probe the CLI once with a cheap entered-competitions request.
v=subprocess.run(['kaggle','--version'],capture_output=True,text=True)
print(v.stdout.strip() or v.stderr.strip())
entered=subprocess.run(['kaggle','competitions','list','--group','entered'],capture_output=True,text=True,timeout=60)
if entered.returncode:
    raise RuntimeError(
        'Kaggle CLI is not authenticated / competition access is unavailable. '
        'Confirm Internet=ON and that you joined Kaggriculture.\n'+entered.stderr[-2000:]
    )

def replay_path(eid):
    return REPLAY_DIR/f'episode-{eid}-replay.json'

def fetch_one(eid):
    p=replay_path(eid)
    if p.exists() and p.stat().st_size>100:
        return {'episode_id':eid,'status':'cached','bytes':p.stat().st_size,'error':''}
    err=''
    for attempt in range(RETRIES):
        r=subprocess.run(
            ['kaggle','competitions','replay',str(eid),'-p',str(REPLAY_DIR)],
            capture_output=True,text=True,timeout=120
        )
        hits=list(REPLAY_DIR.glob(f'*{eid}*replay*.json'))
        if hits and hits[0].stat().st_size>100:
            return {'episode_id':eid,'status':'ok','bytes':hits[0].stat().st_size,'error':''}
        err=(r.stderr or r.stdout)[-1200:]
        time.sleep(1.0+attempt)
    return {'episode_id':eid,'status':'failed','bytes':0,'error':err}

rows=[]
with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as ex:
    fut={ex.submit(fetch_one,eid):eid for eid in chosen}
    for i,f in enumerate(as_completed(fut),1):
        rows.append(f.result())
        if i%25==0:
            pd.DataFrame(rows).to_csv(WORK/'replay_download_manifest.csv',index=False)
            ok=sum(r['status'] in ('ok','cached') for r in rows)
            print(f'{i}/{len(chosen)} downloaded/cached={ok}')

manifest=pd.DataFrame(rows)
manifest.to_csv(WORK/'replay_download_manifest.csv',index=False)
display(manifest.status.value_counts())
files=sorted(REPLAY_DIR.glob('*replay*.json'))
if not files:
    display(manifest[manifest.status=='failed'].head(20))
    raise RuntimeError('No replay downloads succeeded.')
print('Replay files:',len(files),'MB=',round(sum(p.stat().st_size for p in files)/2**20,1))


In [ ]:
# -------- Parallel replay parsing --------
# We write one Parquet shard per replay. This avoids returning large DataFrames
# through multiprocessing IPC and lets a failed run resume from completed shards.
from joblib import Parallel, delayed
from src.kagv2.replay import load_replay,replay_to_turn_frame,add_outcome_labels,add_future_opponent_sell_labels
from src.kagv2.features import daily_macro_frame

SHARDS=WORK/'turn_shards'
SHARDS.mkdir(exist_ok=True)
N_JOBS=max(1,min(4,(os.cpu_count() or 2)-1))

def parse_shard(path_str):
    p=Path(path_str)
    out=SHARDS/(p.stem+'.parquet')
    if out.exists() and out.stat().st_size>200:
        return str(out), 'cached', 0
    try:
        df=replay_to_turn_frame(load_replay(p),p,stride=1)
        df=add_outcome_labels(df)
        df=add_future_opponent_sell_labels(df,horizon=24)
        df.to_parquet(out,index=False)
        return str(out),'ok',len(df)
    except Exception as e:
        return str(p),'failed',repr(e)[:500]

parsed=Parallel(n_jobs=N_JOBS,prefer='processes',batch_size=2)(
    delayed(parse_shard)(str(p)) for p in files
)
parse_manifest=pd.DataFrame(parsed,columns=['path','status','rows_or_error'])
parse_manifest.to_csv(WORK/'replay_parse_manifest.csv',index=False)
display(parse_manifest.status.value_counts())

shards=sorted(SHARDS.glob('*.parquet'))
if not shards: raise RuntimeError('No replay shards parsed successfully.')
turns=pd.concat((pd.read_parquet(p) for p in shards),ignore_index=True)
turns.to_parquet(WORK/'turns.parquet',index=False)
daily=daily_macro_frame(turns)
daily.to_parquet(WORK/'daily_macros.parquet',index=False)
print('turns',turns.shape,'daily',daily.shape,'episodes',turns.episode_id.nunique())


In [ ]:
# -------- Ladder forensics in the same pass --------
from src.kagv2.ladder import deduplicated_matchups,bradley_terry,open_loop_report

match=deduplicated_matchups(turns)
match.to_parquet(WORK/'matchups.parquet',index=False)
bt=bradley_terry(match)
bt.to_csv(WORK/'bt_strength.csv',index=False)
ol=open_loop_report(turns,min_episodes=3)
ol.to_csv(WORK/'open_loop_report.csv',index=False)

summary={
    'repo_commit':commit,
    'episodes_requested':len(chosen),
    'replays_usable':len(files),
    'episodes_parsed':int(turns.episode_id.nunique()),
    'turn_rows':int(len(turns)),
    'daily_rows':int(len(daily)),
    'matchups':int(len(match)),
    'actors_bt':int(len(bt)),
    'actors_open_loop':int(len(ol)),
}
(WORK/'warehouse_summary.json').write_text(json.dumps(summary,indent=2))
print(json.dumps(summary,indent=2))
display(bt.head(25))
display(ol.head(25))


## Save this notebook version/output

The next primary notebook consumes the output Dataset from this run. The important files are:

`turns.parquet`, `daily_macros.parquet`, `bt_strength.csv`, `open_loop_report.csv`, `matchups.parquet`, `repo_commit.txt`.

You do **not** need to rerun replay downloads for later experiments.
